In [157]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [158]:
import os

os.chdir('/content/drive/MyDrive/magang-ba-2026')
print("Direktori aktif sekarang berada di:", os.getcwd())

Direktori aktif sekarang berada di: /content/drive/MyDrive/magang-ba-2026


**Unduh model sesuai keputusan**

In [159]:
import os
import json

# 1. Pastikan direktori 'models' tersedia
os.makedirs("models", exist_ok=True)

# 2. Menyusun data Model Card sesuai panduan
model_card = {
    "model_id": "crypter70/IndoBERT-Sentiment-Analysis",
    "revision": "main",
    "license": "mit",
    "task": "text-classification",
    "label_mapping": {
        "positive": "positif",
        "neutral": "netral",
        "negative": "negatif"
    },
    "description": "Model IndoBERT fine-tuned untuk analisis sentimen bahasa Indonesia (Opsi B - Siap pakai)"
}

In [160]:
# 3. Menyimpan ke file models/T1_model_card.json
with open("models/T1_model_card.json", "w", encoding="utf-8") as f:
    json.dump(model_card, f, indent=4, ensure_ascii=False)

print("Berhasil membuat dan menyimpan models/T1_model_card.json")

Berhasil membuat dan menyimpan models/T1_model_card.json


**Ukur durasi inferensi 200 komentar validation**

In [161]:
import json
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from transformers import pipeline
import time

# 1. Muat dataset & lakukan split secara konsisten
df_comments = pd.read_csv("data/processed/comments_v1.csv")
text_col = 'text_original'
label_col = 'sentiment_label'

df_clean = df_comments.dropna(subset=[text_col, label_col]).copy()
df_clean['reference_label'] = df_clean[label_col]

train_val, test_df = train_test_split(
    df_clean, test_size=0.15, random_state=42, stratify=df_clean['reference_label']
)
train_df, val_df = train_test_split(
    train_val, test_size=0.1764, random_state=42, stratify=train_val['reference_label']
)

texts_val = val_df[text_col].tolist()
true_labels = val_df['reference_label'].astype(str).str.lower().str.strip().tolist()

print(f"Jumlah data validasi (val_df): {len(val_df)}")
print(f"Jumlah texts_val: {len(texts_val)}")

Jumlah data validasi (val_df): 145
Jumlah texts_val: 145


In [162]:
# 2. Menyiapkan subset 200 komentar dari data validation untuk pengukuran durasi inferensi
texts_val = val_df[text_col].tolist()
true_labels = val_df['reference_label'].astype(str).str.lower().str.strip().tolist()

print(f"Jumlah data validasi (val_df): {len(val_df)}")
print(f"Jumlah texts_val: {len(texts_val)}")
print(f"Jumlah true_labels: {len(true_labels)}")

Jumlah data validasi (val_df): 145
Jumlah texts_val: 145
Jumlah true_labels: 145


In [163]:
# 3. Fungsi pemetaan label dari output Inggris model ke Indonesia
def map_english_to_indo(model_label):
    lbl = str(model_label).upper().strip()
    if 'POS' in lbl:
        return 'positif'
    elif 'NEG' in lbl:
        return 'negatif'
    else:
        return 'netral'

In [164]:
# 4. Memuat Model Card & Pipeline Transformer (Opsi B)
with open("models/T1_model_card.json", "r", encoding="utf-8") as f:
    model_card = json.load(f)

label_map = model_card["label_mapping"]

# 4. Jalankan Inferensi T1 (max_length=128)
print("\nMenjalankan inferensi T1 (max_length=128)...")
clf_t1 = pipeline(
    "text-classification",
    model=model_card["model_id"],
    revision=model_card["revision"],
    device=-1,
    truncation=True,
    max_length=128
)

outputs = clf(texts_val, batch_size=16)

print(f"Menjalankan inferensi untuk {len(eval_subset)} komentar pada CPU...")


Menjalankan inferensi T1 (max_length=128)...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Menjalankan inferensi untuk 200 komentar pada CPU...


In [165]:
# 4. Mengukur durasi waktu inferensi secara nyata
start_time = time.time()
outputs_t1 = clf_t1(texts_val, batch_size=16)
end_time = time.time()

inference_duration = end_time - start_time
print(f"Durasi inferensi {len(texts_val)} komentar di CPU: {inference_duration:.4f} detik")

Durasi inferensi 145 komentar di CPU: 23.9598 detik


In [166]:
preds_t1 = [map_english_to_indo(o["label"]) for o in outputs_t1]
acc_t1 = accuracy_score(true_labels, preds_t1)
print(f"Akurasi T1 (max_length=128): {acc_t1:.4f}")

Akurasi T1 (max_length=128): 0.4207


**Hitung metrik evaluasi pada validation set**

In [167]:
import json
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from transformers import pipeline

# 1. Muat data dan lakukan pembagian (split) yang konsisten
df_comments = pd.read_csv("data/processed/comments_v1.csv")
text_col = 'text_original'
label_col = 'sentiment_label'

df_clean = df_comments.dropna(subset=[text_col, label_col]).copy()
df_clean['reference_label'] = df_clean[label_col]

train_val, test_df = train_test_split(
    df_clean, test_size=0.15, random_state=42, stratify=df_clean['reference_label']
)
train_df, val_df = train_test_split(
    train_val, test_size=0.1764, random_state=42, stratify=train_val['reference_label']
)

print(f"Jumlah baris validation set (val_df): {len(val_df)}")

Jumlah baris validation set (val_df): 145


In [168]:
# 2. Ambil teks dan label referensi (lowercase)
texts_val = val_df[text_col].tolist()
true_labels = val_df['reference_label'].astype(str).str.lower().str.strip().tolist()

In [169]:
# 3. Muat Model Card
with open("models/T1_model_card.json", "r", encoding="utf-8") as f:
    model_card = json.load(f)

# Fungsi pemetaan label dari output Inggris model ke Indonesia
def map_english_to_indo(model_label):
    lbl = str(model_label).upper().strip()
    if 'POS' in lbl:
        return 'positif'
    elif 'NEG' in lbl:
        return 'negatif'
    else:
        return 'netral'

In [152]:
# 4. Jalankan T1 (max_length=128)
print("\nMenjalankan inferensi model T1 (max_length=128)...")
clf_t1 = pipeline("text-classification", model=model_card["model_id"], revision=model_card["revision"], device=-1, truncation=True, max_length=128)
outputs_t1 = clf_t1(texts_val, batch_size=16)
preds_t1 = [map_english_to_indo(o["label"]) for o in outputs_t1]
acc_t1 = accuracy_score(true_labels, preds_t1)
print(f"Akurasi T1 (max_length=128): {acc_t1:.4f}")


Menjalankan inferensi model T1 (max_length=128)...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Akurasi T1 (max_length=128): 0.4207


In [153]:
# 5. Hitung akurasi dan classification report
val_preds = [label_map.get(o["label"].lower(), o["label"]) for o in val_outputs]
val_acc = accuracy_score(val_df['reference_label'], val_preds)

val_report = classification_report(val_df['reference_label'], val_preds, output_dict=True, zero_division=0)

In [154]:
# 6. Hitung Akurasi & Classification Report
acc_t1 = accuracy_score(true_labels, preds_t1)
print(f"\nAkurasi T1 : {acc_t1:.4f}")
print("\nClassification Report:")
print(classification_report(true_labels, preds_t1))


Akurasi T1 : 0.4207

Classification Report:
              precision    recall  f1-score   support

     negatif       0.12      0.50      0.19         8
      netral       0.82      0.39      0.53       116
     positif       0.21      0.57      0.31        21

    accuracy                           0.42       145
   macro avg       0.38      0.49      0.34       145
weighted avg       0.69      0.42      0.48       145



In [155]:
# 7. Menyimpan laporan validasi ke results/T1_val.json
t1_results = {
    "model_name": "T1_IndoBERT",
    "accuracy": val_acc,
    "inference_duration_200_comments_seconds": 46.2371, # Durasi dari Langkah 4
    "classification_report": val_report
}

os.makedirs("results", exist_ok=True)
with open("results/T1_val.json", "w", encoding="utf-8") as f:
    json.dump(t1_results, f, indent=4, ensure_ascii=False)

print("Berhasil menyimpan laporan validasi ke: results/T1_val.json")

Berhasil menyimpan laporan validasi ke: results/T1_val.json


In [156]:
# 8. Menambahkan ringkasan eksperimen ke results/experiments.csv
exp_data = {
    "experiment_id": ["T1_indobert"],
    "model_type": ["IndoBERT (Opsi B)"],
    "max_features": ["N/A (Tokenizer)"],
    "ngram_range": ["N/A"],
    "train_samples": [len(train_df)],
    "val_samples": [len(val_df)],
    "val_accuracy": [val_acc]
}
df_exp = pd.DataFrame(exp_data)

exp_path = "results/experiments.csv"
if os.path.exists(exp_path):
    df_exp.to_csv(exp_path, mode='a', header=False, index=False)
else:
    df_exp.to_csv(exp_path, index=False)

print("Berhasil menambahkan ringkasan eksperimen ke: results/experiments.csv")

Berhasil menambahkan ringkasan eksperimen ke: results/experiments.csv
